# Anima + WAI-Anima — ComfyUI Colab

Один ноутбук для Anima Aesthetic v1.1 и WAI-Anima v1.0. Устанавливает ComfyUI, ComfyUI-Manager, Anima-LLLite и готовые T2I/ControlNet/Inpaint workflow. Токены берутся из Colab Secrets (`HF_TOKEN`, `CIVITAI_API_TOKEN`) или запрашиваются интерактивно.

In [ ]:
# @title 1) Tokens and paths
import os, getpass
from pathlib import Path
try:
    from google.colab import userdata
except Exception:
    userdata = None

def secret(name):
    value = ''
    if userdata is not None:
        try:
            value = userdata.get(name) or ''
        except Exception:
            value = ''
    if isinstance(value, dict):
        value = value.get('value') or value.get('token') or next(iter(value.values()), '')
    if not isinstance(value, str):
        value = str(value) if value else ''
    return value.strip() or os.environ.get(name, '').strip()

HF_TOKEN = secret('HF_TOKEN') or secret('HUGGINGFACE_TOKEN')
CIVITAI_API_TOKEN = secret('CIVITAI_API_TOKEN')
if not HF_TOKEN: HF_TOKEN = getpass.getpass('Hugging Face token (required): ').strip()
if not CIVITAI_API_TOKEN: CIVITAI_API_TOKEN = getpass.getpass('Civitai API token (recommended): ').strip()
if not HF_TOKEN: raise RuntimeError('HF_TOKEN is required.')
os.environ.update({'HF_TOKEN': HF_TOKEN, 'HUGGINGFACE_TOKEN': HF_TOKEN})
if CIVITAI_API_TOKEN: os.environ['CIVITAI_API_TOKEN'] = CIVITAI_API_TOKEN
COMFY_ROOT = Path('/content/ComfyUI'); MODEL_ROOT = COMFY_ROOT / 'models'
print('Tokens configured without displaying their values.')


In [ ]:
# @title 2) Install ComfyUI, Manager and Anima nodes
import subprocess
def run(cmd):
    print('+', cmd); subprocess.run(cmd, shell=True, check=True)
if not COMFY_ROOT.exists(): run('git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI')
run('pip install -q -r /content/ComfyUI/requirements.txt')
NODES = {'ComfyUI-Manager':'https://github.com/ltdrdata/ComfyUI-Manager.git','ComfyUI-Workflow-Models-Downloader':'https://github.com/slahiri/ComfyUI-Workflow-Models-Downloader.git','ComfyUI-Anima-LLLite':'https://github.com/kohya-ss/ComfyUI-Anima-LLLite.git','comfyui_controlnet_aux':'https://github.com/Fannovel16/comfyui_controlnet_aux.git','comfyui-lora-manager':'https://github.com/willmiao/ComfyUI-Lora-Manager.git','rgthree-comfy':'https://github.com/rgthree/rgthree-comfy.git','was-node-suite-comfyui':'https://github.com/WASasquatch/was-node-suite-comfyui.git','ComfyUI-Image-Saver':'https://github.com/alexopus/ComfyUI-Image-Saver.git'}
for folder, repo in NODES.items():
    target = COMFY_ROOT/'custom_nodes'/folder
    if not target.exists(): run(f'git clone --depth 1 {repo} {target}')
    req = target/'requirements.txt'
    if req.exists(): run(f'pip install -q -r {req}')
print('ComfyUI and Anima node set are ready.')

In [ ]:
# @title 3) Download both Anima checkpoints and dependencies
import requests
def download(url, target, headers=None):
    target=Path(target); target.parent.mkdir(parents=True,exist_ok=True)
    if target.exists() and target.stat().st_size>1024: print('exists:',target); return
    h={'Authorization':f'Bearer {HF_TOKEN}'} if 'huggingface.co' in url else {}
    if headers: h.update(headers)
    with requests.get(url,headers=h,stream=True,timeout=60) as r:
        r.raise_for_status()
        with open(target,'wb') as f:
            for chunk in r.iter_content(1024*1024):
                if chunk: f.write(chunk)
    print('downloaded:',target)
def civitai(url,target):
    h={'Authorization':f'Bearer {CIVITAI_API_TOKEN}'} if CIVITAI_API_TOKEN else {}
    download(url,target,h)
download('https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/text_encoders/qwen_3_06b_base.safetensors',MODEL_ROOT/'text_encoders/qwen_3_06b_base.safetensors')
download('https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/vae/qwen_image_vae.safetensors',MODEL_ROOT/'vae/qwen_image_vae.safetensors')
download('https://huggingface.co/Comfy-Org/Anima-LLLite/resolve/main/model_patches/anima-lllite-any-test-like-v2.safetensors',MODEL_ROOT/'model_patches/anima-lllite-any-test-like-v2.safetensors')
download('https://huggingface.co/Comfy-Org/Anima-LLLite/resolve/main/model_patches/anima-lllite-inpainting-v2.safetensors',MODEL_ROOT/'model_patches/anima-lllite-inpainting-v2.safetensors')
civitai('https://civitai.red/api/download/models/3126581?fileId=3007030',MODEL_ROOT/'diffusion_models/anima/anima_aestheticV11.safetensors')
civitai('https://civitai.red/api/download/models/2983680?fileId=2863158',MODEL_ROOT/'diffusion_models/anima/waiANIMA_v10Base10.safetensors')
print('Both checkpoints are available in ComfyUI.')

In [ ]:
# @title 5) Launch ComfyUI + resilient tunnel (Cloudflare -> localhost.run)
import os, queue, re, shutil, socket, subprocess, threading, time
from pathlib import Path

import requests

LOW_VRAM_STABLE = False  # True: slower but safer for large images on free Colab
COMFY_ROOT = Path(globals().get('COMFY_ROOT', '/content/ComfyUI'))
OUTPUT_DIR = COMFY_ROOT / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def stop_process(proc):
    if proc is None or proc.poll() is not None:
        return
    proc.terminate()
    try:
        proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        proc.kill()


# Safe rerun: stop processes and supervisor created by the previous cell run.
old_stop = globals().get('_TUNNEL_STOP')
if old_stop is not None:
    old_stop.set()
stop_process(globals().get('_TUNNEL_PROC'))
stop_process(globals().get('_COMFY_PROC'))
stop_process(globals().get('tunnel'))  # process name used by the previous cell
stop_process(globals().get('comfy'))   # process name used by the previous cell
old_log = globals().get('_COMFY_LOG')
if old_log is not None:
    try:
        old_log.close()
    except Exception:
        pass


def ensure_cloudflared():
    if shutil.which('cloudflared') is not None:
        return True
    try:
        package = Path('/tmp/cloudflared-linux-amd64.deb')
        response = requests.get(
            'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb',
            timeout=90,
        )
        response.raise_for_status()
        package.write_bytes(response.content)
        subprocess.run(['dpkg', '-i', str(package)], check=True)
    except Exception as error:
        print('cloudflared install failed; localhost.run remains available:', error)
    return shutil.which('cloudflared') is not None


ensure_cloudflared()

comfy_args = [
    'python', 'main.py', '--listen', '0.0.0.0', '--port', '8188',
    '--enable-cors-header', '*', '--output-directory', str(OUTPUT_DIR),
]
comfy_args += (
    ['--novram', '--disable-smart-memory', '--cache-none', '--force-upcast-attention']
    if LOW_VRAM_STABLE else ['--lowvram', '--preview-method', 'auto']
)
_COMFY_LOG = open('/content/comfyui.log', 'a', encoding='utf-8', buffering=1)
_COMFY_PROC = subprocess.Popen(
    comfy_args, cwd=COMFY_ROOT, stdout=_COMFY_LOG, stderr=subprocess.STDOUT,
)


def wait_for_comfy(timeout=240):
    deadline = time.time() + timeout
    while time.time() < deadline:
        if _COMFY_PROC.poll() is not None:
            raise RuntimeError('ComfyUI exited. Inspect /content/comfyui.log')
        try:
            with socket.create_connection(('127.0.0.1', 8188), timeout=2):
                response = requests.get('http://127.0.0.1:8188/system_stats', timeout=5)
                if response.ok:
                    return
        except (OSError, requests.RequestException):
            time.sleep(2)
    raise TimeoutError('ComfyUI did not become ready within 240 seconds.')


wait_for_comfy()
print('ComfyUI is ready locally. Output:', OUTPUT_DIR)


def read_process_lines(proc, line_queue):
    for line in iter(proc.stdout.readline, ''):
        line_queue.put(line.rstrip())


def start_and_find_url(command, url_pattern, timeout=75):
    proc = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    lines = queue.Queue()
    threading.Thread(target=read_process_lines, args=(proc, lines), daemon=True).start()
    recent = []
    deadline = time.time() + timeout
    while time.time() < deadline and proc.poll() is None:
        try:
            line = lines.get(timeout=1)
        except queue.Empty:
            continue
        recent = (recent + [line])[-10:]
        match = re.search(url_pattern, line, re.I)
        if match:
            return proc, match.group(0), recent
    stop_process(proc)
    return None, None, recent


def start_cloudflare():
    if shutil.which('cloudflared') is None:
        return None, None, ['cloudflared is not installed']
    return start_and_find_url(
        [
            'cloudflared', 'tunnel', '--no-autoupdate',
            '--url', 'http://127.0.0.1:8188', '--protocol', 'http2',
            '--edge-ip-version', '4', '--loglevel', 'info',
        ],
        r'https://[-a-z0-9]+\.trycloudflare\.com',
    )


def start_localhost_run():
    if shutil.which('ssh') is None:
        return None, None, ['OpenSSH client is not installed']
    return start_and_find_url(
        [
            'ssh', '-T',
            '-o', 'StrictHostKeyChecking=no',
            '-o', 'UserKnownHostsFile=/dev/null',
            '-o', 'ServerAliveInterval=30',
            '-o', 'ServerAliveCountMax=3',
            '-o', 'ExitOnForwardFailure=yes',
            '-R', '80:127.0.0.1:8188', 'nokey@localhost.run',
        ],
        r'https://[-a-z0-9]+\.(?:lhr\.life|localhost\.run)',
    )


_TUNNEL_STOP = threading.Event()
_TUNNEL_PROC = None
_CLOUDFLARE_BLOCKED_UNTIL = 0.0


def tunnel_supervisor():
    global _TUNNEL_PROC, _CLOUDFLARE_BLOCKED_UNTIL
    while not _TUNNEL_STOP.is_set():
        proc = url = None
        recent = []

        if time.time() >= _CLOUDFLARE_BLOCKED_UNTIL:
            print('Trying Cloudflare Quick Tunnel once...')
            proc, url, recent = start_cloudflare()
            joined = '\n'.join(recent).lower()
            if '1015' in joined or '429 too many requests' in joined:
                _CLOUDFLARE_BLOCKED_UNTIL = time.time() + 300
                print('Cloudflare rate limit detected. Cooldown: 5 minutes.')

        if proc is None:
            print('Using localhost.run fallback...')
            proc, url, fallback_recent = start_localhost_run()
            recent += fallback_recent

        if proc is None:
            print('No tunnel available. Recent output:', recent[-4:])
            _TUNNEL_STOP.wait(60)
            continue

        _TUNNEL_PROC = proc
        print('Open ComfyUI:', url)

        # Do not probe the public URL repeatedly. It can trigger provider limits.
        while not _TUNNEL_STOP.wait(10):
            if _COMFY_PROC.poll() is not None:
                print('ComfyUI stopped. Inspect /content/comfyui.log')
                stop_process(proc)
                return
            if proc.poll() is not None:
                print('Tunnel process exited; reconnecting...')
                break
        stop_process(proc)


threading.Thread(target=tunnel_supervisor, daemon=True).start()
print('Tunnel supervisor started in background. This cell may finish normally.')
